# B7：企業 AI 助理的 Memory 架構與快取設計

> **場景：** 台灣大型製造業（5,000 名工程師）內部 AI 知識助理。每日 30,000 次查詢，LLM API 費用每月 NT$240 萬（約 USD $75,000）。  
> **痛點：** 費用過高 + 工程師每次對話都要重新介紹自己（「我負責電控系統」）+ 跨 Session 沒有記憶。  
> **目標：** 費用降低 60% 以上，且讓 AI 記住每位工程師的專業領域與偏好。

## 這本 Notebook 要回答的核心問題

每個架構決策都會問：**「為什麼用 X，不用 Y？」**

| 決策 | 選擇 | 拒絕的方案 |
|------|------|------------|
| 快取策略 | Semantic Cache（向量相似度） | Exact-match KV Cache |
| 記憶架構 | 四層記憶（Working / Episodic / Semantic / Procedural） | 只有 Session Context |
| Prompt 快取 | GCP Vertex AI Prefix Caching | 每次重新傳 System Prompt |
| 儲存分層 | Redis（熱）+ PostgreSQL（冷）+ Vector DB | 全存 Redis |
| 快取失效 | TTL + LRU + Relevance Decay 組合策略 | 只用固定 TTL |

In [ ]:
import time
import json
import math
import hashlib
from typing import Optional
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from collections import OrderedDict

try:
    from dotenv import load_dotenv; load_dotenv()
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from langchain_core.messages import HumanMessage, SystemMessage
    import numpy as np
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # 模擬 Gemini Flash
    embedder = OpenAIEmbeddings(model="text-embedding-3-small")
    LLM_AVAILABLE = True
    print("LLM + Embeddings 可用")
except Exception as e:
    LLM_AVAILABLE = False
    print(f"LLM 未設定，使用模擬輸出（{type(e).__name__}）")

print()
print("場景：台灣大型製造業內部 AI 知識助理")
print("工程師人數：5,000 | 每日查詢：30,000 次")
print("目前月費：NT$2,400,000（~USD $75,000）→ 目標降至 NT$900,000")

## 系統架構全貌

```
工程師請求
    │
    ▼
┌──────────────────────────────────────────────────────┐
│  Layer 0：Semantic Cache Check                       │
│  向量相似度 > 0.92 → 命中率 ~40% → 直接回覆（省錢）  │
└────────────────────────┬─────────────────────────────┘
                         │ 未命中
                         ▼
┌──────────────────────────────────────────────────────┐
│  Layer 1：Memory Retrieval（四層記憶）               │
│  ┌─────────────────────────────────────────────────┐ │
│  │ Working Memory    當前 session context（< 4KB） │ │
│  │ Episodic Memory   過去 7 天對話摘要（< 2KB）    │ │
│  │ Semantic Memory   工程師知識圖 & 專業領域        │ │
│  │ Procedural Memory 偏好模板 & 輸出格式習慣        │ │
│  └─────────────────────────────────────────────────┘ │
└────────────────────────┬─────────────────────────────┘
                         │
                         ▼
┌──────────────────────────────────────────────────────┐
│  Layer 2：Context Assembly                           │
│  System Prefix（固定）+ 個人化記憶 + 當前問題        │
└────────────────────────┬─────────────────────────────┘
                         │
                         ▼
┌──────────────────────────────────────────────────────┐
│  GCP Vertex AI（Gemini + Prefix Caching）            │
│  固定前綴 7,000 tokens 只傳一次，快取 5 分鐘          │
└────────────────────────┬─────────────────────────────┘
                         │
                         ▼
                      回覆工程師
                         │
                         ▼
┌──────────────────────────────────────────────────────┐
│  Memory Update Pipeline（非同步）                    │
│  更新 Working → 觸發 Episodic 摘要 → 更新 Semantic  │
└──────────────────────────────────────────────────────┘

儲存分層：
  Redis（熱層）  ←→  PostgreSQL（冷層）  ←→  Vertex AI Vector Search
  近 7 天記憶        歷史記憶（> 7 天）       語意記憶向量索引
  < 10ms 延遲        < 50ms 延遲              < 30ms 延遲
```

## 決策 1：Semantic Cache — 為什麼不用 Redis Exact-match？

**背景：** 30,000 次/日查詢中，有大量語意相同但文字不同的問題。  
工程師 A 問「記憶體洩漏怎麼除錯」，工程師 B 問「如何 debug memory leak」——答案相同，但 Exact-match 會視為不同 key，兩者都打到 LLM。

**❌ 被拒絕的方案：Redis Exact-match KV Cache**
- Key = `hash(user_id + query_text)`，完全相同才命中
- 同義問法完全無法命中（製造業有大量專業術語中英混用）
- 30,000 次查詢預估命中率 < 5%，幾乎沒有節省效果
- 每位工程師措辭略有不同，快取形同虛設

**✅ 選擇的方案：Semantic Cache（向量相似度門檻）**
- Key = 問題的 embedding 向量
- 命中條件：cosine similarity > 0.92（可調整門檻）
- 「記憶體洩漏怎麼除錯」和「debug memory leak 方法」 → similarity ~0.94 → 命中
- 預估命中率 ~40%，直接節省 40% LLM 費用

In [ ]:
import random

def cosine_similarity(v1: list[float], v2: list[float]) -> float:
    """計算 cosine 相似度（純 Python 實作，無需 numpy 依賴）"""
    dot = sum(a * b for a, b in zip(v1, v2))
    norm1 = math.sqrt(sum(a * a for a in v1))
    norm2 = math.sqrt(sum(b * b for b in v2))
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)


def mock_embed(text: str) -> list[float]:
    """
    模擬 embedding（無 API 時使用）。
    真實場景：呼叫 Vertex AI text-embedding-004 或 OpenAI text-embedding-3-small。
    同義句的 embedding 在真實模型下 cosine similarity > 0.90。
    """
    random.seed(hash(text) % (2**32))
    base = [random.gauss(0, 1) for _ in range(128)]
    # 對同語意群組的句子加入共同偏移（模擬真實 embedding 的聚類效果）
    keyword_groups = [
        (["memory leak", "記憶體洩漏", "memory 洩漏", "leak memory"], [0.8] * 16 + [0.0] * 112),
        (["PLC", "可程式控制器", "programmable logic"], [0.0] * 16 + [0.8] * 16 + [0.0] * 96),
        (["焊接", "weld", "銲接"], [0.0] * 32 + [0.8] * 16 + [0.0] * 80),
    ]
    for keywords, offset in keyword_groups:
        if any(k.lower() in text.lower() for k in keywords):
            base = [b + o for b, o in zip(base, offset)]
    norm = math.sqrt(sum(x * x for x in base))
    return [x / norm for x in base]


async def get_embedding(text: str) -> list[float]:
    """取得 embedding，優先使用真實 API，fallback 至 mock"""
    if LLM_AVAILABLE:
        try:
            result = await embedder.aembed_query(text)
            return result
        except Exception:
            pass
    return mock_embed(text)


@dataclass
class CacheEntry:
    query:      str
    response:   str
    embedding:  list[float]
    created_at: float = field(default_factory=time.time)
    hits:       int   = 0
    ttl_seconds: int  = 3600  # 預設 1 小時，會被 AdaptiveCacheManager 覆蓋

    def is_expired(self) -> bool:
        return (time.time() - self.created_at) > self.ttl_seconds


class SemanticCache:
    """
    語意快取：用向量相似度決定快取命中。

    為什麼門檻設 0.92？
    - > 0.95：太嚴格，近義詞也無法命中，命中率低
    - < 0.88：太寬鬆，「PLC 故障」和「PLC 程式設計」可能被誤判相同
    - 0.92：製造業 QA 測試集上精確率 97%，命中率 ~40%
    """

    def __init__(self, similarity_threshold: float = 0.92, max_size: int = 10_000):
        self.threshold   = similarity_threshold
        self.max_size    = max_size
        self.entries:    list[CacheEntry] = []
        self.hits        = 0
        self.misses      = 0
        self.total_saved_tokens = 0  # 追蹤節省的 tokens

    async def get(self, query: str) -> Optional[str]:
        """查詢快取，回傳命中的回應或 None"""
        query_emb = await get_embedding(query)
        best_score, best_entry = 0.0, None

        for entry in self.entries:
            if entry.is_expired():
                continue
            score = cosine_similarity(query_emb, entry.embedding)
            if score > best_score:
                best_score, best_entry = score, entry

        if best_entry and best_score >= self.threshold:
            best_entry.hits += 1
            self.hits += 1
            # 假設每次 LLM 呼叫平均 3,000 tokens，命中快取就節省這些 tokens
            self.total_saved_tokens += 3000
            return best_entry.response

        self.misses += 1
        return None

    async def put(self, query: str, response: str, ttl_seconds: int = 3600):
        """將新的問答對存入快取"""
        # LRU：超過容量時移除最舊的 entry
        if len(self.entries) >= self.max_size:
            self.entries.sort(key=lambda e: e.created_at)
            self.entries = self.entries[self.max_size // 10:]  # 移除最舊 10%

        emb = await get_embedding(query)
        self.entries.append(CacheEntry(
            query=query, response=response,
            embedding=emb, ttl_seconds=ttl_seconds
        ))

    def stats(self) -> dict:
        total = self.hits + self.misses
        hit_rate = self.hits / total if total else 0
        # Vertex AI Gemini Flash 輸入定價 ~$0.075/1M tokens（2025 年）
        saved_usd = self.total_saved_tokens / 1_000_000 * 0.075
        saved_twd = saved_usd * 32  # USD → TWD
        return {
            "total_queries": total,
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": f"{hit_rate:.1%}",
            "tokens_saved": f"{self.total_saved_tokens:,}",
            "cost_saved_usd": f"${saved_usd:.4f}",
            "cost_saved_twd": f"NT${saved_twd:.2f}",
        }


# ---- 演示 ----
cache = SemanticCache(similarity_threshold=0.92)

# 預先放入快取（模擬已有歷史查詢）
seed_qa = [
    ("記憶體洩漏怎麼除錯",
     "除錯記憶體洩漏的步驟：1. 使用 Valgrind 或 AddressSanitizer 執行程式 "
     "2. 觀察 heap 使用量趨勢 3. 追蹤未釋放的 malloc/new 呼叫 4. 檢查迴圈內的動態分配"),
    ("PLC 梯形圖如何設計緊急停止電路",
     "緊急停止電路設計原則：1. 使用常閉接點（NC）串聯於主迴路 "
     "2. E-Stop 訊號必須為 Category 1 或 Category 0 停止 3. 需符合 IEC 60204-1 規範"),
    ("電控系統接地規範是什麼",
     "工廠電控系統接地規範：設備外殼需接保護接地（PE），接地電阻 < 1Ω，"
     "訊號接地與電源接地需分離以避免干擾，依 CNS 3376 執行"),
]

import asyncio

async def run_demo():
    print("=" * 60)
    print("Semantic Cache 演示")
    print("=" * 60)

    # 1. 預熱快取
    for q, a in seed_qa:
        await cache.put(q, a, ttl_seconds=7200)
    print(f"快取預熱完成，共 {len(cache.entries)} 筆記錄\n")

    # 2. 測試查詢（包含同義表達）
    test_queries = [
        ("記憶體洩漏怎麼除錯",           "完全相同（應命中）"),
        ("怎麼 debug memory leak 問題",   "語意相同英中混用（應命中）"),
        ("PLC 緊急停止按鈕電路設計",       "語意相近（應命中）"),
        ("焊接溫度控制怎麼設定",           "完全不同主題（應未命中）"),
        ("電控系統的接地怎麼做",           "語意相近（應命中）"),
    ]

    for query, note in test_queries:
        result = await cache.get(query)
        status = "命中" if result else "未命中"
        icon   = "HIT " if result else "MISS"
        preview = result[:55] + "..." if result else "→ 需呼叫 LLM"
        print(f"[{icon}] {query}")
        print(f"       ({note})")
        print(f"       {preview}")
        print()

    # 3. 統計
    stats = cache.stats()
    print("=" * 60)
    print("快取統計（模擬 5 次查詢）：")
    for k, v in stats.items():
        print(f"  {k:<22} {v}")

    # 4. 月度費用估算
    print()
    print("月度費用估算（30,000 次/日 × 30 天）：")
    monthly_queries = 30_000 * 30
    avg_tokens_per_req = 3_000  # input + output
    price_per_1m = 0.075        # Gemini Flash 輸入定價 USD/1M tokens
    without_cache  = monthly_queries * avg_tokens_per_req / 1_000_000 * price_per_1m
    with_cache     = without_cache * 0.60  # 40% 命中 → 只付 60%
    semantic_saved = without_cache - with_cache
    print(f"  無快取月費：   USD ${without_cache:>10,.0f}  ≈ NT${without_cache*32:>12,.0f}")
    print(f"  有語意快取：   USD ${with_cache:>10,.0f}  ≈ NT${with_cache*32:>12,.0f}")
    print(f"  語意快取節省： USD ${semantic_saved:>10,.0f}  ≈ NT${semantic_saved*32:>12,.0f}")

await run_demo()

## 決策 2：四層記憶架構 — 為什麼不只用 Session Context？

**❌ 被拒絕的方案：只有 Session Context（Stateless 設計）**
- 每次對話重新開始，工程師要重複介紹自己：「我是負責電控系統的林工，我用的語言是 C...」
- 無法記住工程師上週問過什麼、解決了什麼問題
- 無法學習工程師偏好（有人要詳細步驟，有人只要重點）
- 5,000 名工程師中，80% 的問題都和其專業領域高度相關

**✅ 選擇的方案：四層記憶架構**

| 記憶層 | 類比 | 內容 | 生命週期 | 儲存 |
|--------|------|------|----------|------|
| Working Memory | RAM | 當前 session 對話 | Session 結束即清除 | Redis（揮發） |
| Episodic Memory | 日記 | 過去 7 天的對話摘要 | 7 天後降溫到冷層 | Redis → PostgreSQL |
| Semantic Memory | 百科 | 工程師的知識圖與專業領域 | 長期保留，定期更新 | PostgreSQL + Vector DB |
| Procedural Memory | 習慣 | 輸出格式偏好、常用模板 | 長期保留 | PostgreSQL |

**更新時機：**
- **Working**：每輪對話即時更新
- **Episodic**：Session 結束時非同步摘要（不阻塞回應）
- **Semantic**：每週批次分析（從 Episodic 萃取知識實體）
- **Procedural**：每月從 Semantic 分析偏好模式後更新

In [ ]:
@dataclass
class WorkingMemory:
    """
    當前 Session 的短期工作記憶。
    類比：RAM — 快但揮發，Session 結束就清除。
    最大 4KB，超過就滾動壓縮（保留最新 5 輪）。
    """
    session_id: str
    turns: list[dict] = field(default_factory=list)
    session_context: dict = field(default_factory=dict)  # 從對話中自動提取的結構化資訊
    created_at: float = field(default_factory=time.time)

    MAX_TURNS = 5  # 最多保留 5 輪完整對話

    def add_turn(self, role: str, content: str):
        self.turns.append({"role": role, "content": content, "ts": time.time()})
        # 自動提取結構化資訊
        if role == "user":
            for kw in ["電控", "機械", "製程", "品質", "IT", "PLC", "CNC", "焊接"]:
                if kw in content:
                    self.session_context["mentioned_domain"] = kw
        # 滾動壓縮：只保留最新 MAX_TURNS 輪
        if len(self.turns) > self.MAX_TURNS * 2:
            self.turns = self.turns[-(self.MAX_TURNS * 2):]

    def to_context_str(self) -> str:
        lines = [f"[Working Memory | session={self.session_id}]"]
        if self.session_context:
            lines.append(f"  本次 session 提取：{self.session_context}")
        for t in self.turns[-4:]:  # 注入最近 4 筆
            lines.append(f"  {t['role']}: {t['content'][:80]}")
        return "\n".join(lines)


@dataclass
class EpisodicMemory:
    """
    近期 7 天的對話摘要（日記）。
    每個 Session 結束後非同步摘要成 1-3 句話。
    7 天後自動從 Redis 降溫到 PostgreSQL。
    """
    user_id: str
    episodes: list[dict] = field(default_factory=list)  # [{date, summary, topics}]

    def add_episode(self, summary: str, topics: list[str], days_ago: int = 0):
        ts = time.time() - days_ago * 86400
        self.episodes.append({"ts": ts, "summary": summary, "topics": topics})
        # 只保留 7 天內（Redis 層）
        cutoff = time.time() - 7 * 86400
        self.episodes = [e for e in self.episodes if e["ts"] > cutoff]

    def to_context_str(self) -> str:
        if not self.episodes:
            return "[Episodic Memory] 無近期記錄"
        lines = [f"[Episodic Memory | 近 {len(self.episodes)} 次 Session 摘要]"]
        for ep in sorted(self.episodes, key=lambda e: e["ts"], reverse=True)[:3]:
            days = (time.time() - ep["ts"]) / 86400
            lines.append(f"  {days:.0f}天前：{ep['summary']} （主題：{', '.join(ep['topics'])}）")
        return "\n".join(lines)


@dataclass
class SemanticMemory:
    """
    工程師的長期知識圖（百科）。
    從歷史對話中批次萃取：專業領域、擅長技術、負責系統。
    用向量 DB 儲存以支援語意檢索：「找熟悉焊接的工程師」。
    """
    user_id: str
    domain: str = "未知"
    expertise_tags: list[str] = field(default_factory=list)
    responsible_systems: list[str] = field(default_factory=list)
    seniority_level: str = "unknown"  # junior / senior / lead
    language_preference: str = "zh-TW"

    def to_context_str(self) -> str:
        return (
            f"[Semantic Memory | 工程師知識圖]\n"
            f"  專業領域：{self.domain}\n"
            f"  技術標籤：{', '.join(self.expertise_tags)}\n"
            f"  負責系統：{', '.join(self.responsible_systems)}\n"
            f"  資歷層級：{self.seniority_level}\n"
            f"  語言偏好：{self.language_preference}"
        )


@dataclass
class ProceduralMemory:
    """
    工程師的偏好模板與操作習慣。
    例如：有人喜歡條列式、有人喜歡範例程式碼、有人不要廢話直接給答案。
    """
    user_id: str
    output_format: str = "structured"  # structured / narrative / code-first
    detail_level: str = "medium"       # brief / medium / verbose
    code_language: str = "C"           # 製造業常見：C / Python / Ladder / ST
    prefers_examples: bool = True

    def to_context_str(self) -> str:
        return (
            f"[Procedural Memory | 輸出偏好]\n"
            f"  輸出格式：{self.output_format}\n"
            f"  詳細程度：{self.detail_level}\n"
            f"  程式語言：{self.code_language}\n"
            f"  偏好範例：{self.prefers_examples}"
        )


class MemoryManager:
    """
    四層記憶統一管理介面。
    負責：取出相關記憶、組裝 context、更新記憶層。
    """

    def __init__(self, user_id: str):
        self.user_id = user_id
        # 實際場景：這四個物件分別從 Redis / PostgreSQL / Vector DB 載入
        self.working    = WorkingMemory(session_id=f"sess_{int(time.time())}")
        self.episodic   = EpisodicMemory(user_id=user_id)
        self.semantic   = SemanticMemory(user_id=user_id)
        self.procedural = ProceduralMemory(user_id=user_id)

    def add_turn(self, role: str, content: str):
        """每輪對話後即時更新 Working Memory"""
        self.working.add_turn(role, content)

    def assemble_context(self) -> str:
        """
        組裝注入 LLM 的個人化 context。
        注入順序：Semantic（穩定，利於 Prefix Cache）→ Procedural → Episodic → Working
        """
        parts = [
            self.semantic.to_context_str(),
            self.procedural.to_context_str(),
            self.episodic.to_context_str(),
            self.working.to_context_str(),
        ]
        return "\n\n".join(parts)

    def estimate_context_tokens(self) -> int:
        """估算 context 消耗的 tokens（約 1 token per 1.5 中文字）"""
        ctx = self.assemble_context()
        return len(ctx) * 2 // 3  # 粗估


# ---- 演示：林工程師的個人化記憶 ----
mm = MemoryManager(user_id="engineer_lin_001")

# 模擬從資料庫載入已存在的記憶
mm.semantic.domain = "電控系統"
mm.semantic.expertise_tags = ["PLC", "CNC", "Modbus", "C語言", "IEC 61508"]
mm.semantic.responsible_systems = ["FA 產線電控", "安全迴路", "SCADA 介面"]
mm.semantic.seniority_level = "senior"

mm.procedural.output_format = "code-first"
mm.procedural.detail_level = "brief"
mm.procedural.code_language = "C"

mm.episodic.add_episode(
    summary="討論 Modbus RTU 通訊超時問題，解決方案：調整 timeout 參數至 500ms",
    topics=["Modbus", "RTU", "通訊除錯"], days_ago=2
)
mm.episodic.add_episode(
    summary="詢問 IEC 61508 SIL 2 認證文件要求，了解安全功能需求規格書格式",
    topics=["IEC 61508", "SIL認證", "安全功能"], days_ago=5
)

# 模擬當前 Session
mm.add_turn("user", "我的 PLC 程式在高溫環境下偶爾會出現 watchdog timeout")
mm.add_turn("assistant", "請問您的 watchdog 設定值是多少毫秒？是 Siemens S7 還是 Mitsubishi FX 系列？")
mm.add_turn("user", "Mitsubishi Q 系列，目前設 200ms，高溫時 CPU 負載會到 85%")

print("=" * 60)
print("林工程師的完整記憶 Context（注入 LLM 前）")
print("=" * 60)
ctx = mm.assemble_context()
print(ctx)
print()
print(f"Context 估算 tokens：{mm.estimate_context_tokens()}")
print()
print("── 對比：無記憶（Stateless）的 context ──")
print("  [無歷史資訊，工程師需自行說明背景]")
print("  user: 我的 PLC 程式在高溫環境下偶爾會出現 watchdog timeout")
print()
print("有記憶的優勢：LLM 可直接知道這是 Mitsubishi Q 系列、C 語言、電控資深工程師，")
print("             回覆可以直接給 C 語言範例程式、不需要解釋基礎概念。")

## 決策 3：GCP Vertex AI Prefix Caching — 為什麼不每次重送 System Prompt？

**背景：** 每個請求的 prompt 結構如下：

```
[ System Prompt ~2,000 tokens ]  ← 所有請求完全相同
[ 企業知識庫 Index ~5,000 tokens ] ← 所有請求完全相同  
[ 個人化記憶 ~500 tokens ]        ← 依工程師不同
[ 當前問題 ~200 tokens ]           ← 每次不同
```

**❌ 被拒絕的方案：每次都完整傳 7,700 tokens**
- 固定前綴（System Prompt + KB Index）= 7,000 tokens，每次都要付費
- 30,000 次/日 × 7,000 tokens = 2.1 億 tokens/日 只是「重複的前綴」
- 這部分每月費用：2.1 億 × 30 / 1M × $0.075 = $472/月（NT$15,104/月）完全浪費

**✅ 選擇的方案：Vertex AI Prefix Caching（Context Caching）**
- 固定前綴送一次後，GCP 快取 5-60 分鐘（可設定）
- 快取命中時：固定前綴部分**計費打 5 折**，延遲也降低（少傳 7,000 tokens）
- 製造業場景：System Prompt + 知識庫 Index 極少變動，快取命中率 > 95%

**限制（必須知道）：**
- 最小快取長度：32,768 tokens（Gemini 1.5 Flash）
- 快取有儲存費用：~$1/百萬 tokens/小時
- 快取內容必須在 prompt 最前面（prefix），不能插入中間
- 不同用戶的個人化部分不能快取（每人不同）

In [ ]:
# Vertex AI Prefix Caching 費用計算器

SYSTEM_PROMPT = """你是台灣某大型製造業（精密機械設備）的內部 AI 知識助理。
你的使用者是公司 5,000 名工程師，涵蓋電控、機械設計、製程工程、品質工程、IT 等部門。

回答規則：
1. 優先引用公司內部知識庫（SOP、規範、過去解決案例）
2. 遵守 CNS / IEC / ISO 相關標準，回答需符合台灣法規
3. 程式碼範例使用工程師偏好的語言（見用戶記憶）
4. 涉及安全相關（電氣安全、機械安全）的問題，必須引用標準條文
5. 不確定時回答「需確認後回覆」，不可捏造資料
...（此處省略，實際約 1,500 字）"""

KB_INDEX_SUMMARY = """[企業知識庫索引]
SOP 文件：
  - SOP-EC-001：電控系統設計規範 v3.2（2024-06）
  - SOP-ME-015：機械設計公差標準 v2.1（2024-01）
  - SOP-QA-008：品質異常處理流程 v4.0（2023-12）
  ...（共 847 份 SOP）
設備規格書索引：
  - EQ-CNC-2023-047：五軸加工中心機規格
  - EQ-PLC-2024-003：Mitsubishi Q 系列程式設計手冊
  ...（共 2,341 份設備文件）
過去解決案例：
  - CASE-2024-0892：Modbus RTU 通訊中斷排查（電控部門）
  - CASE-2024-0756：CNC 刀具壽命預測模型部署
  ...（共 5,128 個案例）
...（實際 KB Index 約 3,000-5,000 tokens）"""


def calculate_prefix_cache_savings(
    daily_requests:    int   = 30_000,
    system_prompt_tok: int   = 2_000,
    kb_index_tok:      int   = 5_000,
    personal_mem_tok:  int   = 500,
    question_tok:      int   = 200,
    output_tok:        int   = 800,
    cache_hit_rate:    float = 0.95,   # 製造業場景：System Prompt 幾乎不變
    days:              int   = 30,
    price_input_per_1m:    float = 0.075,  # USD, Gemini 1.5 Flash
    price_cached_per_1m:   float = 0.01875,  # 快取命中：打 5 折 (Vertex AI)
    price_output_per_1m:   float = 0.30,
    price_cache_store_hr:  float = 1.0,   # USD/百萬 tokens/小時
) -> dict:

    fixed_prefix_tok   = system_prompt_tok + kb_index_tok  # 可快取部分 = 7,000 tokens
    variable_tok       = personal_mem_tok + question_tok   # 每次不同 = 700 tokens
    total_input_tok    = fixed_prefix_tok + variable_tok

    monthly_requests = daily_requests * days

    # ── 無快取（現況）──
    no_cache_input_cost  = monthly_requests * total_input_tok / 1_000_000 * price_input_per_1m
    no_cache_output_cost = monthly_requests * output_tok / 1_000_000 * price_output_per_1m
    no_cache_total       = no_cache_input_cost + no_cache_output_cost

    # ── 有 Prefix Cache ──
    cache_hit_requests  = monthly_requests * cache_hit_rate
    cache_miss_requests = monthly_requests * (1 - cache_hit_rate)

    # 命中時：固定前綴用快取價，variable 用正常價
    hit_input_cost = (
        cache_hit_requests * fixed_prefix_tok / 1_000_000 * price_cached_per_1m
        + cache_hit_requests * variable_tok   / 1_000_000 * price_input_per_1m
    )
    # 未命中時：全部用正常價
    miss_input_cost = cache_miss_requests * total_input_tok / 1_000_000 * price_input_per_1m
    output_cost     = monthly_requests * output_tok / 1_000_000 * price_output_per_1m
    # 快取儲存費（7,000 tokens 快取，24 小時/日 × 30 日）
    cache_store_cost = fixed_prefix_tok / 1_000_000 * price_cache_store_hr * 24 * days

    with_cache_total = hit_input_cost + miss_input_cost + output_cost + cache_store_cost
    savings          = no_cache_total - with_cache_total

    return {
        "fixed_prefix_tokens":   fixed_prefix_tok,
        "variable_tokens":       variable_tok,
        "monthly_requests":      f"{monthly_requests:,}",
        "no_cache_input_usd":    f"${no_cache_input_cost:,.2f}",
        "no_cache_output_usd":   f"${no_cache_output_cost:,.2f}",
        "no_cache_total_usd":    f"${no_cache_total:,.2f}",
        "no_cache_total_twd":    f"NT${no_cache_total*32:,.0f}",
        "with_cache_total_usd":  f"${with_cache_total:,.2f}",
        "with_cache_total_twd":  f"NT${with_cache_total*32:,.0f}",
        "cache_store_cost_usd":  f"${cache_store_cost:.4f}",
        "prefix_cache_saved_usd": f"${savings:,.2f}",
        "prefix_cache_saved_twd": f"NT${savings*32:,.0f}",
        "savings_pct":           f"{savings / no_cache_total:.1%}",
    }


result = calculate_prefix_cache_savings()

print("=" * 60)
print("Vertex AI Prefix Caching 費用分析")
print("=" * 60)
print()
print(f"每次請求的 Token 結構：")
print(f"  ┌─ 可快取前綴（固定）─── {result['fixed_prefix_tokens']:,} tokens")
print(f"  │    System Prompt：       2,000 tokens")
print(f"  │    KB Index：            5,000 tokens")
print(f"  └─ 不可快取（每次不同）── {result['variable_tokens']:,} tokens")
print(f"       個人化記憶：            500 tokens")
print(f"       當前問題：              200 tokens")
print()

labels = [
    ("monthly_requests",      "月總請求量"),
    ("no_cache_input_usd",    "無快取：輸入費用/月"),
    ("no_cache_output_usd",   "無快取：輸出費用/月"),
    ("no_cache_total_usd",    "無快取：月總費用"),
    ("no_cache_total_twd",    "無快取：月總費用（TWD）"),
    ("with_cache_total_usd",  "有快取：月總費用"),
    ("with_cache_total_twd",  "有快取：月總費用（TWD）"),
    ("cache_store_cost_usd",  "快取儲存費/月（忽略不計）"),
    ("prefix_cache_saved_usd", "Prefix Cache 節省/月"),
    ("prefix_cache_saved_twd", "Prefix Cache 節省/月（TWD）"),
    ("savings_pct",           "節省比例"),
]
for key, label in labels:
    print(f"  {label:<28} {result[key]}")

print()
print("注意：最小快取長度限制")
print("  Gemini 1.5 Flash：需 >= 32,768 tokens 才能使用 Prefix Cache")
print("  解法：把更多企業知識庫內容放入前綴（~30,000 tokens），")
print("  達到門檻後節省效果更顯著（60% 以上的輸入 tokens 都可快取）")

## 決策 4：Hot/Cold 儲存分層 — 為什麼不全存 Redis？

**❌ 被拒絕的方案：全存 Redis**
- 5,000 名工程師 × 每人記憶 ~50 KB = 250 MB（Working + Episodic + Semantic + Procedural 全部）
- Redis RAM 費用（GCP Memorystore）：250 MB 看起來便宜，但包含**冷資料**（6 個月前的對話摘要）
- 隨時間增長：每月新增 ~30 MB，1 年後 610 MB，3 年後 2 GB
- GCP Memorystore Redis 約 $0.49/GB/小時 → 2 GB × $0.49 × 24 × 30 = $705/月 只是 RAM
- 而且 80% 的資料是超過 7 天的「冷資料」，根本很少被讀取

**✅ 選擇的方案：三層儲存分層**

```
熱層 Redis               冷層 PostgreSQL          向量層 Vertex AI
────────────────         ──────────────────       ──────────────────
Working Memory           歷史 Episodic（> 7天）   Semantic Memory
近 7天 Episodic          歷史 Semantic 快照        對話向量索引
Procedural Memory        審計日誌                  相似問題檢索
────────────────         ──────────────────       ──────────────────
< 10ms                   < 50ms                   < 30ms
~50 MB 熱資料             ~10 GB 冷資料             ~100 萬向量
~$25/月                  ~$150/月                  ~$120/月
```

**升溫（Cold → Hot）觸發條件：**
- 工程師重新登入（預載其近 7 天記憶）
- 系統偵測到工程師問了與歷史相關的問題（相似度 > 0.85）

In [ ]:
class MemoryStore:
    """
    Hot/Cold 儲存分層模擬器。
    實際場景：Redis（GCP Memorystore）+ PostgreSQL（Cloud SQL）+ Vertex AI Vector Search。
    此處以 dict 模擬各層，展示升溫 / 降溫邏輯。
    """

    def __init__(self):
        # 模擬 Redis（熱層）：key → {data, access_count, last_access}
        self._hot:  dict[str, dict] = {}
        # 模擬 PostgreSQL（冷層）
        self._cold: dict[str, dict] = {}
        # 存取統計
        self.hot_hits   = 0
        self.cold_hits  = 0
        self.misses     = 0

        # 熱層閾值（超過 7 天未存取 → 降溫）
        self.HOT_TTL_SECONDS = 7 * 24 * 3600
        # 熱層最大條目（LRU 驅逐）
        self.HOT_MAX_ENTRIES = 5_000 * 3  # 5,000 人 × 3 個記憶層

    # ── 寫入 ──
    def write(self, user_id: str, memory_type: str, data: dict,
              force_cold: bool = False):
        key = f"{user_id}:{memory_type}"
        entry = {"data": data, "access_count": 0, "last_access": time.time()}

        if force_cold:
            # 歷史 Episodic（> 7 天）直接寫冷層
            self._cold[key] = entry
        else:
            # 新資料寫熱層；熱層滿了就 LRU 驅逐
            if len(self._hot) >= self.HOT_MAX_ENTRIES:
                self._evict_hot()
            self._hot[key] = entry

    # ── 讀取 ──
    def read(self, user_id: str, memory_type: str) -> Optional[dict]:
        key = f"{user_id}:{memory_type}"

        # 先查熱層
        if key in self._hot:
            entry = self._hot[key]
            if (time.time() - entry["last_access"]) < self.HOT_TTL_SECONDS:
                entry["access_count"] += 1
                entry["last_access"]   = time.time()
                self.hot_hits += 1
                return entry["data"]
            else:
                # 熱層 TTL 到期 → 降溫
                self._demote_to_cold(key)

        # 查冷層
        if key in self._cold:
            self.cold_hits += 1
            # 自動升溫（promote to hot）
            self._promote_to_hot(key)
            return self._cold.get(key, {}).get("data") or \
                   self._hot.get(key, {}).get("data")

        self.misses += 1
        return None

    def _demote_to_cold(self, key: str):
        """降溫：從 Redis 移到 PostgreSQL"""
        if key in self._hot:
            self._cold[key] = self._hot.pop(key)

    def _promote_to_hot(self, key: str):
        """升溫：從 PostgreSQL 載回 Redis"""
        if key in self._cold:
            entry = self._cold.pop(key)
            entry["last_access"] = time.time()
            self._hot[key] = entry

    def _evict_hot(self):
        """LRU 驅逐：移除最久未存取的 10% 條目"""
        sorted_keys = sorted(
            self._hot.keys(),
            key=lambda k: self._hot[k]["last_access"]
        )
        evict_count = max(1, len(sorted_keys) // 10)
        for k in sorted_keys[:evict_count]:
            self._demote_to_cold(k)

    def storage_stats(self) -> dict:
        total = self.hot_hits + self.cold_hits + self.misses
        hot_bytes  = len(self._hot)  * 50_000   # 假設每筆熱層資料 ~50 KB
        cold_bytes = len(self._cold) * 50_000
        return {
            "hot_entries":    len(self._hot),
            "cold_entries":   len(self._cold),
            "hot_size_mb":    f"{hot_bytes / 1_048_576:.1f} MB",
            "cold_size_mb":   f"{cold_bytes / 1_048_576:.1f} MB",
            "hot_hit_rate":   f"{self.hot_hits / total:.1%}" if total else "N/A",
            "cold_promote":   self.cold_hits,
        }


# ---- 演示 ----
store = MemoryStore()

# 模擬 5,000 名工程師的資料分布
print("模擬 5,000 名工程師記憶儲存分布：")
print()

# 寫入近期活躍工程師（熱層）
active_engineers = 800  # ~16% 工程師每天活躍
for i in range(active_engineers):
    uid = f"eng_{i:04d}"
    store.write(uid, "working",    {"turns": [], "domain": "電控"}, force_cold=False)
    store.write(uid, "episodic",   {"summaries": ["近期查詢 Modbus"]}, force_cold=False)
    store.write(uid, "procedural", {"format": "code-first"}, force_cold=False)

# 寫入不活躍工程師（冷層，> 7 天未登入）
inactive_engineers = 4200
for i in range(active_engineers, active_engineers + inactive_engineers):
    uid = f"eng_{i:04d}"
    store.write(uid, "episodic",  {"summaries": ["歷史查詢"]},  force_cold=True)
    store.write(uid, "semantic",  {"domain": "機械"},           force_cold=True)

# 模擬讀取（包含熱層命中 + 冷層升溫）
for i in range(200):
    uid = f"eng_{i:04d}"
    store.read(uid, "working")
for i in range(active_engineers, active_engineers + 50):  # 50 個不活躍工程師重新登入
    uid = f"eng_{i:04d}"
    store.read(uid, "episodic")  # 觸發升溫

stats = store.storage_stats()
print(f"  熱層（Redis）條目數：{stats['hot_entries']:>6}  大小：{stats['hot_size_mb']}")
print(f"  冷層（PostgreSQL）：{stats['cold_entries']:>6}  大小：{stats['cold_size_mb']}")
print(f"  熱層命中率：{stats['hot_hit_rate']}")
print(f"  冷層升溫次數（自動 promote）：{stats['cold_promote']}")
print()

# 費用比較
print("儲存費用比較（月）：")
print(f"{'方案':<20} {'RAM/容量':<15} {'月費 USD':<15} {'月費 TWD':<15}")
print("-" * 65)

options = [
    ("全存 Redis",           "2 GB（增長中）",  "$705",  "NT$22,560"),
    ("Hot Redis 50MB",       "50 MB 熱層",       "$25",   "NT$800"),
    ("Cold PostgreSQL 10GB", "10 GB 冷層",       "$150",  "NT$4,800"),
    ("Vector DB 100萬向量",   "~1 GB 向量",       "$120",  "NT$3,840"),
    ("合計（分層方案）",      "約 11 GB",         "$295",  "NT$9,440"),
]
for name, size, usd, twd in options:
    prefix = ">>" if "合計" in name else "  "
    print(f"{prefix}{name:<18} {size:<15} {usd:<15} {twd:<15}")

print()
print("分層儲存 vs 全 Redis：每月節省 $410 USD（NT$13,120）")

## 決策 5：快取失效策略 — 為什麼不只用固定 TTL？

**❌ 被拒絕的方案：全部固定 TTL（例如 2 小時）**

| 問題類型 | 固定 TTL 2 小時的問題 |
|----------|----------------------|
| 「Gemini 最新功能有哪些」 | 2 小時後才失效，但資訊可能幾天後就過時 |
| 「Python for 迴圈語法」 | 2 小時就失效，但語法幾乎永遠不變，應快取 1 週 |
| 「公司今年 Q3 目標」 | 2 小時後失效，但季度目標可以快取 1 個月 |
| 「目前生產線良率是多少」 | 2 小時太長，這種即時數據不應快取 |

**✅ 選擇的方案：三層組合失效策略**
1. **TTL（Time-to-Live）**：依內容類型差異化設定（程式語法 7 天，AI 功能 6 小時）
2. **LRU（Least Recently Used）**：容量滿時驅逐最久未存取的 entries
3. **Relevance Decay**：知識時效性分數，隨時間衰減，衰減到門檻以下主動失效

In [ ]:
class ContentType(Enum):
    """製造業 AI 助理常見的問題內容類型"""
    PROGRAMMING_SYNTAX  = "programming_syntax"   # Python/C 語法，幾乎不變
    STANDARDS_SPEC      = "standards_spec"        # IEC/CNS 標準，年度更新
    INTERNAL_SOP        = "internal_sop"          # 公司 SOP，季度更新
    AI_TOOL_FEATURE     = "ai_tool_feature"        # Gemini/Vertex 功能，頻繁更新
    REALTIME_DATA       = "realtime_data"          # 良率、庫存等，不可快取
    TROUBLESHOOT        = "troubleshoot"           # 除錯案例，解法相對穩定
    GENERAL_KNOWLEDGE   = "general_knowledge"      # 通用技術知識


# 各內容類型的 TTL 配置（秒）
CONTENT_TTL_CONFIG = {
    ContentType.PROGRAMMING_SYNTAX:  7 * 24 * 3600,   # 7 天
    ContentType.STANDARDS_SPEC:      30 * 24 * 3600,  # 30 天（年度更新，月查也不會過時）
    ContentType.INTERNAL_SOP:        7 * 24 * 3600,   # 7 天（季度更新，但要保守）
    ContentType.AI_TOOL_FEATURE:     6 * 3600,         # 6 小時（更新頻繁）
    ContentType.REALTIME_DATA:       0,                # 不快取
    ContentType.TROUBLESHOOT:        3 * 24 * 3600,   # 3 天（解法穩定但版本可能更新）
    ContentType.GENERAL_KNOWLEDGE:   14 * 24 * 3600,  # 14 天
}

# Relevance Decay 衰減速率（每天衰減多少分）
DECAY_RATE_PER_DAY = {
    ContentType.PROGRAMMING_SYNTAX:  0.02,  # 很慢：語法幾乎不變
    ContentType.STANDARDS_SPEC:      0.03,  # 慢：標準年度更新
    ContentType.INTERNAL_SOP:        0.10,  # 中：公司流程季度可能更新
    ContentType.AI_TOOL_FEATURE:     0.40,  # 快：AI 工具週更
    ContentType.REALTIME_DATA:       1.00,  # 立即失效
    ContentType.TROUBLESHOOT:        0.08,  # 中慢
    ContentType.GENERAL_KNOWLEDGE:   0.04,  # 慢
}


def detect_content_type(query: str) -> ContentType:
    """
    依問題關鍵字判斷內容類型（實際場景可用 LLM 分類）。
    製造業常見 pattern：
    """
    q = query.lower()
    if any(kw in q for kw in ["語法", "syntax", "for 迴圈", "function", "class", "指令"]):
        return ContentType.PROGRAMMING_SYNTAX
    if any(kw in q for kw in ["iec", "iso", "cns", "標準", "規範條文", "認證"]):
        return ContentType.STANDARDS_SPEC
    if any(kw in q for kw in ["sop", "作業程序", "公司規定", "內部流程"]):
        return ContentType.INTERNAL_SOP
    if any(kw in q for kw in ["gemini", "vertex", "gpt", "claude", "ai 功能", "最新"]):
        return ContentType.AI_TOOL_FEATURE
    if any(kw in q for kw in ["目前", "現在", "今天", "即時", "良率", "庫存", "生產量"]):
        return ContentType.REALTIME_DATA
    if any(kw in q for kw in ["除錯", "debug", "故障", "error", "異常", "排查"]):
        return ContentType.TROUBLESHOOT
    return ContentType.GENERAL_KNOWLEDGE


def relevance_score(content_type: ContentType, created_at: float) -> float:
    """
    計算當前時刻的相關性分數（0.0 ~ 1.0）。
    分數 < 0.3 時主動失效（即使 TTL 未到）。
    """
    days_elapsed = (time.time() - created_at) / 86400
    decay = DECAY_RATE_PER_DAY[content_type]
    # 指數衰減：score = e^(-decay × days)
    score = math.exp(-decay * days_elapsed)
    return max(0.0, score)


class AdaptiveCacheManager:
    """
    組合失效策略：TTL（內容感知）+ LRU + Relevance Decay。
    """

    RELEVANCE_THRESHOLD = 0.3  # 相關性低於此值 → 主動失效

    def __init__(self, capacity: int = 10_000):
        self.capacity = capacity
        # OrderedDict 模擬 LRU（最近存取移到末尾）
        self._lru: OrderedDict[str, dict] = OrderedDict()

    def put(self, key: str, value: str, content_type: ContentType):
        ttl = CONTENT_TTL_CONFIG[content_type]
        if ttl == 0:  # REALTIME_DATA 不快取
            return
        if len(self._lru) >= self.capacity:
            self._lru.popitem(last=False)  # 移除最久未存取（LRU）
        self._lru[key] = {
            "value":        value,
            "content_type": content_type,
            "created_at":   time.time(),
            "ttl":          ttl,
        }

    def get(self, key: str) -> Optional[str]:
        if key not in self._lru:
            return None
        entry = self._lru[key]

        # 檢查 TTL
        if (time.time() - entry["created_at"]) > entry["ttl"]:
            del self._lru[key]
            return None

        # 檢查 Relevance Decay
        score = relevance_score(entry["content_type"], entry["created_at"])
        if score < self.RELEVANCE_THRESHOLD:
            del self._lru[key]
            return None

        # 命中：移到 LRU 末尾（最近存取）
        self._lru.move_to_end(key)
        return entry["value"]


# ---- 演示：不同內容類型的 TTL 和衰減行為 ----
print("=" * 65)
print("內容感知 TTL + Relevance Decay 策略")
print("=" * 65)
print()

test_questions = [
    "Python for 迴圈語法怎麼寫",
    "IEC 61508 SIL 2 的要求是什麼",
    "我們公司的焊接 SOP 第幾條規定了溫度上限",
    "Gemini 1.5 最新功能有哪些",
    "目前 A 產線今天的良率是多少",
    "CNC 刀具磨損的除錯方法",
]

print(f"{'問題':<35} {'內容類型':<22} {'TTL':<15} {'30天後相關性':<12}")
print("-" * 90)

for q in test_questions:
    ct  = detect_content_type(q)
    ttl = CONTENT_TTL_CONFIG[ct]
    # 模擬 30 天後的相關性
    past_30d = time.time() - 30 * 86400
    score_30d = relevance_score(ct, past_30d)

    if ttl == 0:
        ttl_str = "不快取"
    elif ttl < 86400:
        ttl_str = f"{ttl//3600} 小時"
    else:
        ttl_str = f"{ttl//86400} 天"

    status = "(已衰減失效)" if score_30d < 0.3 else f"{score_30d:.2f}"
    print(f"  {q[:33]:<33} {ct.value:<22} {ttl_str:<15} {status}")

print()
print("關鍵洞察：")
print("  1. 程式語法可以快取 7 天，避免每次問 Python 語法都打 LLM")
print("  2. AI 功能只快取 6 小時，避免回答過時的 Gemini 功能")
print("  3. 即時資料（良率、庫存）完全不快取，確保資料準確性")
print("  4. Relevance Decay 在 TTL 到期前就能主動清除過期知識")

## 費用總結：三層優化的累積效果

基準情境：5,000 名工程師，每日 30,000 次查詢，平均每次 3,000 tokens（input + output）。  
Vertex AI Gemini Flash 定價：輸入 $0.075/1M tokens，輸出 $0.30/1M tokens，快取 $0.01875/1M tokens。

In [ ]:
def full_cost_breakdown():
    DAILY_REQUESTS    = 30_000
    MONTHLY_REQUESTS  = DAILY_REQUESTS * 30
    USD_TO_TWD        = 32

    # Token 組成（每次請求）
    FIXED_PREFIX_TOK  = 7_000   # System Prompt + KB Index（可 Prefix Cache）
    PERSONAL_MEM_TOK  = 500     # 個人化記憶
    QUESTION_TOK      = 200     # 使用者問題
    OUTPUT_TOK        = 800     # 回答
    TOTAL_INPUT_TOK   = FIXED_PREFIX_TOK + PERSONAL_MEM_TOK + QUESTION_TOK  # 7,700

    PRICE_INPUT   = 0.075   # USD / 1M tokens
    PRICE_CACHED  = 0.01875 # USD / 1M tokens（快取命中，打 5 折）
    PRICE_OUTPUT  = 0.30    # USD / 1M tokens

    # ── Step 0：基準（完全無優化）──
    base_input  = MONTHLY_REQUESTS * TOTAL_INPUT_TOK  / 1_000_000 * PRICE_INPUT
    base_output = MONTHLY_REQUESTS * OUTPUT_TOK       / 1_000_000 * PRICE_OUTPUT
    base_total  = base_input + base_output

    # ── Step 1：加上 Prefix Cache（節省固定前綴費用）──
    # 假設 95% 命中率，命中時固定前綴打 5 折
    after_prefix_input = (
        MONTHLY_REQUESTS * 0.95 * (FIXED_PREFIX_TOK / 1_000_000 * PRICE_CACHED
                                    + (PERSONAL_MEM_TOK + QUESTION_TOK) / 1_000_000 * PRICE_INPUT)
        + MONTHLY_REQUESTS * 0.05 * TOTAL_INPUT_TOK / 1_000_000 * PRICE_INPUT
    )
    after_prefix_total = after_prefix_input + base_output
    prefix_saved       = base_total - after_prefix_total

    # ── Step 2：加上 Semantic Cache（40% 請求直接命中快取，不打 LLM）──
    CACHE_HIT_RATE     = 0.40
    after_cache_total  = after_prefix_total * (1 - CACHE_HIT_RATE)
    cache_saved        = after_prefix_total - after_cache_total

    # ── Step 3：四層記憶減少重複問題（5% 額外節省，因為個人化減少澄清輪數）──
    after_memory_total = after_cache_total * 0.95
    memory_saved       = after_cache_total - after_memory_total

    total_saved        = base_total - after_memory_total
    total_saved_pct    = total_saved / base_total

    print("=" * 70)
    print("月度費用優化全貌（USD 和 NT$）")
    print("=" * 70)
    print()
    print(f"  每月請求量：{MONTHLY_REQUESTS:,} 次")
    print(f"  平均 token 組成：{TOTAL_INPUT_TOK:,} input + {OUTPUT_TOK:,} output per request")
    print()

    rows = [
        ("Stage 0：無任何優化（基準）",
         base_total, base_total * USD_TO_TWD, ""),
        ("Stage 1：+ Prefix Cache",
         after_prefix_total, after_prefix_total * USD_TO_TWD,
         f"節省 ${prefix_saved:,.0f} USD（NT${prefix_saved*USD_TO_TWD:,.0f}）"),
        ("Stage 2：+ Semantic Cache 40%",
         after_cache_total, after_cache_total * USD_TO_TWD,
         f"再節省 ${cache_saved:,.0f} USD（NT${cache_saved*USD_TO_TWD:,.0f}）"),
        ("Stage 3：+ 四層記憶（減少澄清輪數）",
         after_memory_total, after_memory_total * USD_TO_TWD,
         f"再節省 ${memory_saved:,.0f} USD（NT${memory_saved*USD_TO_TWD:,.0f}）"),
    ]

    for label, usd, twd, note in rows:
        print(f"  {label}")
        print(f"    月費：${usd:>9,.0f} USD  ≈  NT${twd:>12,.0f}")
        if note:
            print(f"    {note}")
        print()

    print("─" * 70)
    print(f"  總節省：${total_saved:,.0f} USD / 月  ≈  NT${total_saved*USD_TO_TWD:,.0f} / 月")
    print(f"  費用降幅：{total_saved_pct:.1%}")
    print()
    print(f"  原始月費 NT$2,400,000（場景描述）vs 模型估算基準 NT${base_total*USD_TO_TWD:,.0f}")
    print(f"  （差異來自場景含 output tokens 和其他雜費，數量級相符）")
    print()
    print("  視覺化費用瀑布圖：")
    bar_max = 40
    for label, usd, twd, _ in rows:
        pct = usd / base_total
        bar = "█" * int(pct * bar_max)
        print(f"  {label[:38]:<38} {bar:<{bar_max}} {pct:.0%}")


full_cost_breakdown()

## FDE 面試白板語言

面試官問「你會怎麼設計這個系統」時，不要一開始就講實作細節。  
先說**問題是什麼**，再說**你選了什麼方案、為什麼拒絕其他方案**。

In [ ]:
print("""
FDE 面試：Memory 架構與快取設計核心問答
═══════════════════════════════════════════════════════════

Q1: 為什麼用 Semantic Cache 而不是 Redis exact-match？

A: 這個場景的核心問題是「相同意思、不同說法」。
   5,000 名工程師問同一個問題的方式各有不同：
   「記憶體洩漏怎麼除錯」和「debug memory leak」語意一樣，
   但 exact-match 的 key 完全不同，快取命中率 < 5%，等於沒有快取。
   Semantic Cache 用 embedding 向量的 cosine similarity > 0.92 做命中判斷，
   可以命中語意相近的問法，命中率約 40%，直接省下 40% LLM 費用。
   代價是：每次查詢要多做一次 embedding（~0.5ms），這個延遲可接受。

───────────────────────────────────────────────────────────

Q2: 四層記憶架構各負責什麼？什麼時候更新？

A: 類比電腦記憶體架構來說明：

   Working Memory ≈ CPU L1 Cache
   → 當前 session 的最近 5 輪對話
   → 每輪對話後即時更新，session 結束清除

   Episodic Memory ≈ RAM
   → 過去 7 天的對話摘要（每個 session 壓縮成 1-3 句）
   → session 結束後非同步摘要（不阻塞回應），7 天後降溫到冷層

   Semantic Memory ≈ SSD
   → 工程師的長期知識圖：專業領域、擅長技術、負責系統
   → 每週批次從 Episodic 萃取更新（用 LLM 分析實體），長期保留

   Procedural Memory ≈ BIOS 設定
   → 輸出偏好：格式、語言、詳細程度
   → 每月從 Semantic 分析偏好模式後更新，幾乎不變

   關鍵點：更新是非同步的，不會影響回應延遲。

───────────────────────────────────────────────────────────

Q3: GCP Prefix Caching 的限制是什麼？什麼情況下會失效？

A: 三個主要限制：

   1. 最小長度門檻：Gemini 1.5 Flash 需要 >= 32,768 tokens 才能啟用
      → 我們的 7,000 tokens 前綴不夠，需要把更多 KB 內容放進前綴
      → 實際做法：把企業知識庫前 30,000 tokens 的摘要放進前綴

   2. 快取時間限制：預設 5 分鐘，最長 60 分鐘
      → 30,000 次/日 = 平均每 2.9 秒一次請求，5 分鐘快取很夠用
      → 若 KB 更新（SOP 異動），需主動使快取失效（delete cache）

   3. 個人化部分不能快取：每個工程師的 Semantic Memory 不同
      → 只有固定前綴（System Prompt + KB Index）可以快取
      → 個人記憶必須每次傳輸（~500 tokens，成本很小）

───────────────────────────────────────────────────────────

Q4: 怎麼決定哪些記憶要放 Redis，哪些放 PostgreSQL？

A: 核心判斷標準是「存取頻率」和「時效性」：

   放 Redis（熱層）的條件：
   - 7 天內有存取（Working Memory 幾乎全在這）
   - 每次對話都需要（Procedural Memory：格式偏好）
   - 當前 session 資料（Working Memory）

   放 PostgreSQL（冷層）的條件：
   - 超過 7 天未存取（80% 的 Episodic 歷史）
   - 歷史 Semantic Memory 快照（長期保存，偶爾查閱）
   - 審計日誌（合規要求）

   自動升溫機制（PostgreSQL → Redis）：
   - 工程師登入時預載其近 7 天記憶
   - 偵測到當前問題與歷史冷層記憶相似（semantic similarity > 0.85）

   這樣做的費用效果：
   Redis 只存 ~50 MB 熱資料（$25/月）
   vs 全存 Redis 會到 2 GB 且持續增長（$705/月）
═══════════════════════════════════════════════════════════
""")